In [1]:
%%writefile ewaste_data.csv
center_id,date,device_type,weight_kg,processed_flag
C001,2023-10-01,Smartphone,50.5,true
C001,2023-10-02,Laptop,120.0,true
C001,2023-10-03,Smartphone,20.0,false
C002,2023-10-01,Tablet,45.0,true
C002,2023-10-01,Tablet,-5.0,false
C002,2023-10-02,Laptop,200.0,true
C003,2023-10-04,Monitor,30.0,false
C003,2023-10-05,Monitor,-10.0,false

Writing ewaste_data.csv


In [2]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.functions import col, sum as _sum, to_date, when
from pyspark.sql.types import DateType

def load_ewaste_data(spark: SparkSession, path: str) -> DataFrame:
    df = spark.read.csv(path, header=True, inferSchema=True)
    return df.withColumn("date", to_date(col("date")))

def filter_valid_weights(df: DataFrame) -> DataFrame:
    return df.filter(col("weight_kg") >= 0)

def aggregate_total_weight(df: DataFrame) -> DataFrame:
    return df.groupBy("center_id", "device_type").agg(
        _sum("weight_kg").alias("total_weight")
    )

def aggregate_processed_weight(df: DataFrame) -> DataFrame:
    return df.filter(col("processed_flag") == True).groupBy("center_id", "device_type").agg(
        _sum("weight_kg").alias("processed_weight")
    )

def with_efficiency(df: DataFrame) -> DataFrame:
    return df.withColumn(
        "efficiency",
        when(col("total_weight") == 0, 0.0)
        .otherwise(col("processed_weight") / col("total_weight"))
    )

In [4]:
spark = SparkSession.builder \
        .appName("EWasteProcessingAnalysis") \
        .getOrCreate()
    
file_path = "ewaste_data.csv"
raw_df = load_ewaste_data(spark, file_path)
valid_df = filter_valid_weights(raw_df)
total_df = aggregate_total_weight(valid_df)
processed_df = aggregate_processed_weight(valid_df)

joined_df = total_df.join(
    processed_df,
    on=["center_id", "device_type"],
    how="left"
).fillna({"processed_weight": 0.0})
final_df = with_efficiency(joined_df)
final_df.orderBy("center_id", "device_type").show()

+---------+-----------+------------+----------------+------------------+
|center_id|device_type|total_weight|processed_weight|        efficiency|
+---------+-----------+------------+----------------+------------------+
|     C001|     Laptop|       120.0|           120.0|               1.0|
|     C001| Smartphone|        70.5|            50.5|0.7163120567375887|
|     C002|     Laptop|       200.0|           200.0|               1.0|
|     C002|     Tablet|        45.0|            45.0|               1.0|
|     C003|    Monitor|        30.0|             0.0|               0.0|
+---------+-----------+------------+----------------+------------------+

